# An equal-budget benchmark: which method, for the same cost?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PINNeAPPle-Labs/PINNeAPPle/blob/main/notebooks/colab/03_equal_budget_benchmark.ipynb)

Comparing methods is only fair when each gets the same budget: the same training steps, the same collocation points,
the same seeds, scored on the same points against the same reference. This notebook compares three network
architectures trained as PINNs and a finite-element solve on a Poisson problem with a known solution, then shows the
error against the budget and each method's trust card.

The equal-budget benchmark API `pp.benchmark` (data preparation, correctness gates, wall-clock or FLOP budgets,
hyper-parameter search with the same effort per method, an advisor) is being merged; the last section switches to it
automatically when it is installed. Everything above it uses `pp.solve` / `pp.compare`, available in every release.

In [ ]:
# Setup: on Colab (or wherever PINNeAPPle is missing) install it from PyPI, then pick the device.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB or importlib.util.find_spec("pinneapple") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pinneapple>=0.7.0"], check=True)

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pinneapple as pp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PINNeAPPle {pp.__version__} on {DEVICE}")

In [ ]:
# Parameters. SMOKE=True shrinks every size so the notebook runs in seconds (the CI runs it that way).
SMOKE = os.environ.get("PINNEAPPLE_NB_SMOKE", "0") == "1"
BUDGET_EPOCHS = 60 if SMOKE else 1000            # full training budget per run (same for every network)
BUDGET_LEVELS = [0.5, 1.0] if SMOKE else [0.25, 0.5, 1.0]  # fractions of the budget (error vs cost curve)
N_COLLOCATION = 256 if SMOKE else 1024
SEEDS = [0] if SMOKE else [0, 1]
ARCHITECTURES = ["modified_mlp", "siren"] if SMOKE else ["modified_mlp", "siren", "vanilla_pinn"]
HIDDEN, LAYERS = (16, 2) if SMOKE else (32, 4)

## The problem: Poisson on the unit square with a manufactured solution

$\nabla^2 u = f$ with $f = -2\pi^2 \sin(\pi x)\sin(\pi y)$ and $u = 0$ on the boundary; the exact solution is
$u = \sin(\pi x)\sin(\pi y)$.

In [ ]:
from pinneapple_physics.pde_environment.builder import ProblemBuilder

spec = (ProblemBuilder("poisson_manufactured").domain(x=(0, 1), y=(0, 1)).fields("u").pde("poisson")
        .bc("dirichlet", field="u", value=0.0, on="x_boundary")
        .bc("dirichlet", field="u", value=0.0, on="y_boundary").build())
problem = pp.PhysicalProblem.from_pde_spec(spec)

def source(X, ctx=None):
    return -2 * np.pi**2 * np.sin(np.pi * X[:, 0]) * np.sin(np.pi * X[:, 1])

def exact(X):
    return (np.sin(np.pi * X[:, 0]) * np.sin(np.pi * X[:, 1]))[:, None]

ctx = {"source_fn": source}
rng = np.random.default_rng(0)
points = rng.random((4096, 2))           # the same scoring points for every method

## Equal budget: every network gets the same epochs, collocation points and seeds

`pp.solve` trains each one; `pp.compare` scores every solution at the same points against the same reference and
records training time and inference cost.

In [ ]:
rows = []
for frac in BUDGET_LEVELS:
    epochs = max(1, int(round(frac * BUDGET_EPOCHS)))
    for arch in ARCHITECTURES:
        for seed in SEEDS:
            sol = pp.solve(problem, "pinn", architecture=arch, epochs=epochs, n_collocation=N_COLLOCATION,
                           hidden_dim=HIDDEN, n_layers=LAYERS, seed=seed, ctx=ctx, device=DEVICE)
            r = pp.compare(problem, [sol], reference="exact", options={"exact": {"fn": exact}}, points=points).rows[0]
            rows.append({"method": f"pinn/{arch}", "budget": frac, "epochs": epochs, "seed": seed,
                         "rel_l2": r["metrics"]["relative_l2"]["u"], "train_s": r["wall_time_s"],
                         "infer_us_per_pt": r["infer_us_per_point"]})
fem = pp.solve(problem, "fem", ctx=ctx, n=16 if SMOKE else 40)
r = pp.compare(problem, [fem], reference="exact", options={"exact": {"fn": exact}}, points=points).rows[0]
rows.append({"method": "fem", "budget": 1.0, "epochs": 0, "seed": 0, "rel_l2": r["metrics"]["relative_l2"]["u"],
             "train_s": r["wall_time_s"], "infer_us_per_pt": r["infer_us_per_point"]})
results = pd.DataFrame(rows)
summary = (results[results["budget"] == 1.0].groupby("method")
           .agg(rel_l2_mean=("rel_l2", "mean"), rel_l2_std=("rel_l2", "std"), train_s=("train_s", "mean"),
                infer_us_per_pt=("infer_us_per_pt", "mean"), runs=("seed", "count"))
           .sort_values("rel_l2_mean"))
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for m, g in results[results["method"] != "fem"].groupby("method"):
    s = g.groupby("epochs")["rel_l2"].agg(["mean", "min", "max"])
    ax.errorbar(s.index, s["mean"], yerr=[s["mean"] - s["min"], s["max"] - s["mean"]], marker="o", capsize=3, label=m)
ax.axhline(results.loc[results["method"] == "fem", "rel_l2"].iloc[0], color="k", ls="--", label="FEM")
ax.set(xscale="log", yscale="log", xlabel="training epochs (same for every network)", ylabel="relative L2 error",
       title="Error against budget (bars: min-max over seeds)")
ax.legend()
plt.show()

## A lower error is not a validated result: the trust card of each winner

The ranking says which method is best for this budget; the trust card says whether that best is good enough to use.
`pp.trust_card` is new after 0.7.0: on an older install the cell loads the same module from the repository.

In [ ]:
from pathlib import Path
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/PINNeAPPle-Labs/PINNeAPPle/main/"

def repo_file(rel):
    roots = [Path(os.environ["PINNEAPPLE_REPO"])] if os.environ.get("PINNEAPPLE_REPO") else []
    for root in roots + [Path.cwd(), *Path.cwd().parents]:
        if (root / rel).is_file() and (root / "pinneapple").is_dir():
            return root / rel
    dest = Path("pinneapple_files") / rel
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

def load_trust_card():
    try:
        return pp.trust_card
    except AttributeError:
        spec_ = importlib.util.spec_from_file_location("field_card", repo_file("pinneapple_analysis/trust/field_card.py"))
        mod = importlib.util.module_from_spec(spec_)
        sys.modules["field_card"] = mod
        spec_.loader.exec_module(mod)
        return mod.trust_card

trust_card = load_trust_card()
g = np.linspace(0, 1, 41)
best_arch = summary.index[summary.index != "fem"][0].split("/")[1]
best = pp.solve(problem, "pinn", architecture=best_arch, epochs=BUDGET_EPOCHS, n_collocation=N_COLLOCATION,
                hidden_dim=HIDDEN, n_layers=LAYERS, seed=0, ctx=ctx, device=DEVICE)
for name, sol, is_ml in ((f"pinn/{best_arch}", best, True), ("fem", fem, False)):
    axes = {"x": g, "y": g} if sol.grid is None else sol.grid      # a grid solution is checked on its own nodes
    AX, AY = np.meshgrid(axes["x"], axes["y"], indexing="ij")
    A = np.stack([AX.ravel(), AY.ravel()], axis=1)
    card = trust_card(axes, sol.predict(A)[:, 0].reshape(AX.shape), name=name, is_ml=is_ml,
                      pde="poisson", source=lambda P: source(P), reference=exact(A)[:, 0].reshape(AX.shape),
                      dirichlet={f: 0.0 for f in ("x_min", "x_max", "y_min", "y_max")})
    print(card, "\n")

## Switch to `pp.benchmark` once it is installed

`pp.benchmark.run` adds what a published comparison needs: prepared and hashed data, correctness gates before any
training, budgets in wall-clock seconds or FLOPs, the same hyper-parameter search effort per method, several budget
levels, seeds, a report and an advisor that says what likely hurt each result. The cell runs only when the API is
present (tracked in the follow-up issue); on older installs it prints how to get it.

In [ ]:
HAS_BENCHMARK_API = importlib.util.find_spec("pinneapple_analysis.benchmark") is not None and hasattr(pp, "ComputeBudget")
if HAS_BENCHMARK_API:
    res = pp.benchmark.run(problems=["burgers"], methods=["pinn", "fno", "deeponet"],
                           budget=pp.ComputeBudget(wall_s=20 if SMOKE else 300), budget_levels=[1 / 3, 1.0],
                           seeds=1 if SMOKE else 3, n_trials=2, smoke=SMOKE, out_dir="benchmark_out")
    print(res.table().to_string())
    for d in res.diagnostics:
        print("-", d.line())
else:
    print("pp.benchmark is not in this install yet; the comparison above uses pp.solve/pp.compare with an equal budget.")